# 📊 Урок 52: Pandas — Фундамент аналізу даних

> **Мета уроку:** Навчитися «думати таблицями» — це фундаментальний крок для будь-якого аналітика або інженера даних. У світі `pandas` кожна проблема зводиться до трансформації колонок і рядків.

---

## 🗺️ Зміст уроку

1. Що таке DataFrame — модель реального світу
2. Архітектура Pandas під капотом
3. Управління пам'яттю та типи даних
4. Завантаження CSV-файлів
5. Огляд даних: head, info, describe
6. Фільтрація: loc та query
7. Групування та агрегації
8. Робота з датами (datetime)
9. Об'єднання датасетів — merge та concat

---

**Датасети, які використовуємо:**
- 🌾 `wfp_food_prices_ukr.csv` — ціни на продукти харчування в Україні (WFP)
- 💱 `exchange-rates_ukr.csv` — курс гривні до долара (FAO)
- 📉 `poverty_ukr.csv` — соціально-економічні індикатори
- 🌍 `global-market-monitor.csv` — глобальний моніторинг продовольчих ринків

---

## 1. Що таке DataFrame — модель реального світу 🌍 <a id='1-dataframe'></a>

### Концепція: таблиця як знімок реальності

У `pandas` **DataFrame** — це двовимірна структура даних, яка логічно є аналогом:
- електронної таблиці Excel
- реляційної таблиці SQL
- JSON-об'єктів у вигляді списку словників

Але думати про DataFrame треба глибше — це **модель реального світу**:

| Реальний світ | DataFrame |
|---|---|
| Одна подія, спостереження, об'єкт | **Рядок (row)** |
| Атрибут або характеристика об'єкта | **Стовпець (column)** |
| Унікальний ідентифікатор об'єкта | **Індекс (index)** |

### Архітектура: DataFrame = словник із Series

```
DataFrame
├── Index (індекс рядків) ← спільний для всіх стовпців
├── Column "date"    → Series (dtype: datetime64)
├── Column "market"  → Series (dtype: object / string)
├── Column "price"   → Series (dtype: float64)
└── Column "region"  → Series (dtype: category)
```

**Series** — базовий будівельний блок. Це одновимірний масив даних, **жорстко пов'язаний з об'єктом `Index`**. Концептуально `Series` нагадує словник Python: мітки індексу — це ключі, дані — це значення.

### Ключова різниця: Series vs DataFrame

| | Series | DataFrame |
|---|---|---|
| Вимірність | 1D (одна колонка) | 2D (таблиця) |
| Однорідність | Всі елементи одного типу | Кожна колонка може мати свій тип |
| `[]` оператор | Вибирає **рядки** | Вибирає **стовпці** |
| `.loc[]` | Один аргумент (рядок) | Два аргументи `[рядки, стовпці]` |
| Data alignment | По мітках рядків | По рядках AND по стовпцях одночасно |

> 💡 **Увага:** `df['a']` поверне стовпець `'a'`. Якщо хочете рядок — використовуйте `df.loc['a']`!

### Автоматичне вирівнювання даних (Data Alignment)

Це фундаментальна особливість Pandas: при операціях над кількома об'єктами дані зіставляються **за мітками індексу**, а не за позицією в масиві. Якщо мітка є в одному об'єкті, але відсутня в іншому — система автоматично підставляє `NaN` (аналог SQL `FULL OUTER JOIN`).

In [ ]:
import pandas as pd
import numpy as np

# --- Демонстрація: Series як словник ---
print("=== Series: одновимірний масив з індексом ===")
prices_series = pd.Series(
    data=[45.5, 38.2, 22.1, 67.8],
    index=['Київ', 'Львів', 'Харків', 'Одеса'],
    name='Ціна хліба (UAH)'
)
print(prices_series)
print(f"\nТип даних: {prices_series.dtype}")
print(f"Ціна у Львові: {prices_series['Львів']} UAH")

In [ ]:
# --- Демонстрація: DataFrame = колекція Series ---
print("=== DataFrame: двовимірна таблиця ===")
data = {
    'місто': ['Київ', 'Львів', 'Харків', 'Одеса'],
    'ціна_хліб': [45.5, 38.2, 22.1, 67.8],
    'ціна_молоко': [34.0, 32.5, 29.0, 38.5],
    'регіон': ['Центр', 'Захід', 'Схід', 'Південь']
}
df_demo = pd.DataFrame(data)
print(df_demo)
print(f"\nТипи: {df_demo.dtypes.to_dict()}")
print(f"\ndf_demo['ціна_хліб'] — це Series:")
print(type(df_demo['ціна_хліб']))

In [ ]:
# --- Демонстрація: автоматичне вирівнювання даних ---
print("=== Автоматичне вирівнювання (Data Alignment) ===")
s1 = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
s2 = pd.Series([100, 200, 300], index=['b', 'c', 'd'])  # 'd' є тільки в s2, 'a' — тільки в s1

print("s1:", s1.to_dict())
print("s2:", s2.to_dict())
print("\ns1 + s2 (автоматичне вирівнювання, аналог FULL OUTER JOIN):")
print(s1 + s2)
print("\n💡 'a' і 'd' → NaN, бо вони є тільки в одному з об'єктів")

---

## 2. Архітектура Pandas під капотом 🏗️ <a id='2-architecture'></a>

Уяви Pandas як **висококорівневу обгортку над низькорівневими масивами**, створену спеціально для аналізу гетерогенних (різнотипних) даних.

```
┌─────────────────────────────────────────┐
│         Рівень I/O                      │  read_csv, read_sql, to_excel...
├─────────────────────────────────────────┤
│    Рівень структур даних (Ядро)         │  Series, DataFrame, Index
├─────────────────────────────────────────┤
│      Рівень типів даних (dtypes)        │  int64, float32, category, datetime64
├─────────────────────────────────────────┤
│  Фундаментальний шар (Backend)          │  NumPy  /  Apache Arrow (PyArrow)
└─────────────────────────────────────────┘
```

### Модель виконання: Eager Execution

На відміну від SQL або Dask, Pandas виконує команди **негайно** (eager execution). Це дає розробнику повний контроль, але вимагає самостійної оптимізації коду.

### Обмеження: In-Memory

Дані повністю завантажуються в оперативну пам'ять. Автор бібліотеки рекомендує мати RAM у **5–10 разів більше**, ніж розмір датасету на диску. При перевищенні ліміту переходять на:
- **Chunking** — обробка по частинах
- **Dask** — паралельні обчислення
- **PySpark** — розподілені кластери

In [ ]:
# --- Перевірка версії та бекенду ---
print(f"Pandas версія: {pd.__version__}")
print(f"NumPy версія: {np.__version__}")

# Перевіримо, чи доступний PyArrow
try:
    import pyarrow as pa
    print(f"PyArrow версія: {pa.__version__} ✅")
except ImportError:
    print("PyArrow не встановлено (встановіть: pip install pyarrow)")

---

## 3. Управління пам'яттю та типи даних 💾 <a id='3-memory'></a>

Оскільки Pandas — це in-memory система, правильний вибір типів даних критично важливий.

### NumPy vs PyArrow — два бекенди

| | NumPy (класичний) | PyArrow (сучасний, Pandas ≥ 2.0) |
|---|---|---|
| Рядки | `object` (неефективно!) | `string[pyarrow]` (компактно) |
| Цілі числа з NaN | конвертує в `float64` 😱 | `int64[pyarrow]` (NaN-safe) |
| Пропущені значення | `np.nan` | `pd.NA` |
| Продуктивність | baseline | значно швидше на рядках |

### Практичні оптимізації

1. **Downcasting** — замість `int64` (8 байт) → `int8` (1 байт), якщо значення в діапазоні -128..127
2. **Category** — для текстових стовпців з низькою кардинальністю (регіони, типи товарів). Економія до 90% пам'яті!
3. **datetime64** замість рядків для дат

In [ ]:
# --- Демонстрація економії пам'яті ---

# Симулюємо невеликий датасет з повторюваними категоріями
n = 10_000
df_mem = pd.DataFrame({
    'регіон': np.random.choice(['Київська', 'Львівська', 'Харківська', 'Одеська', 'Дніпропетровська'], n),
    'товар': np.random.choice(['Хліб', 'Молоко', 'Картопля', 'Цибуля'], n),
    'ціна': np.random.uniform(5, 200, n).astype('float64'),
    'кількість': np.random.randint(1, 100, n).astype('int64')
})

print("=== ДО оптимізації ===")
print(df_mem.dtypes)
mem_before = df_mem.memory_usage(deep=True).sum()
print(f"\nПам'ять: {mem_before:,} байт ({mem_before / 1024:.1f} KB)")

In [ ]:
# --- Оптимізуємо типи ---
df_opt = df_mem.copy()

# Текстові колонки з повторами → category
df_opt['регіон'] = df_opt['регіон'].astype('category')
df_opt['товар'] = df_opt['товар'].astype('category')

# Ціна: float32 замість float64 (вдвічі менше)
df_opt['ціна'] = df_opt['ціна'].astype('float32')

# Кількість: int8 (0-100 поміщається в int8, але int16 безпечніше)
df_opt['кількість'] = df_opt['кількість'].astype('int16')

print("=== ПІСЛЯ оптимізації ===")
print(df_opt.dtypes)
mem_after = df_opt.memory_usage(deep=True).sum()
print(f"\nПам'ять: {mem_after:,} байт ({mem_after / 1024:.1f} KB)")
print(f"\n💾 Економія: {(1 - mem_after/mem_before)*100:.1f}%")

---

## 4. Завантаження CSV-файлів 📂 <a id='4-loading'></a>

### `pd.read_csv()` — I/O Readers

Pandas абстрагує від складнощів парсингу. Але **думати таблицями треба вже на етапі завантаження** — правильні параметри зекономлять пам'ять та час.

| Параметр | Призначення |
|---|---|
| `sep` | Розділювач (`,`, `;`, `\t`) |
| `usecols` | Завантажити лише потрібні стовпці |
| `parse_dates` | Автоматично розпізнати дати |
| `dtype` | Вказати типи заздалегідь (уникнути зайвого перетворення) |
| `nrows` | Завантажити лише N рядків (для попереднього огляду) |

> 🎯 **Правило:** Завантажуй лише ті колонки, що потрібні (`usecols`). Для датасетів >1 млн рядків це критично.

In [ ]:
import os

# Шлях до директорії з даними (відносно ноутбука)
DATA_DIR = 'data'

# --- 1. Завантажуємо ціни на продукти харчування ---
# Спочатку подивимося, скільки рядків у файлі
df_food_preview = pd.read_csv(os.path.join(DATA_DIR, 'wfp_food_prices_ukr.csv'), nrows=3)
print("=== Перші 3 рядки (preview) ===")
print(df_food_preview)
print(f"\nКолонки: {list(df_food_preview.columns)}")

In [ ]:
# --- Завантажуємо повний датасет, але тільки потрібні колонки ---
# Нам потрібні: дата, регіон, ринок, категорія товару, товар, ціна в UAH
FOOD_COLS = ['date', 'admin1', 'market', 'category', 'commodity', 'unit', 'currency', 'price']

df_food = pd.read_csv(
    os.path.join(DATA_DIR, 'wfp_food_prices_ukr.csv'),
    usecols=FOOD_COLS,
    parse_dates=['date']  # автоматично розпізнаємо дати
)

print(f"Завантажено: {df_food.shape[0]:,} рядків × {df_food.shape[1]} стовпців")
print(f"\nПеріод даних: {df_food['date'].min().date()} — {df_food['date'].max().date()}")
df_food.head()

In [ ]:
# --- Завантажуємо курс гривні ---
df_rates = pd.read_csv(
    os.path.join(DATA_DIR, 'exchange-rates_ukr.csv'),
    usecols=['StartDate', 'Year', 'Months', 'Value'],
    parse_dates=['StartDate']
)
print(df_rates.columns)
# Перейменовуємо колонки для зручності
df_rates.columns = ['date', 'year', 'month', 'uah_per_usd']
print(f"Курс UAH/USD: {df_rates.shape[0]} записів")
df_rates.head()

---

## 5. Огляд даних: head, info, describe 🔍 <a id='5-explore'></a>

### Принцип: спочатку зрозумій, потім маніпулюй

Три методи, які треба запускати на **кожному** новому датасеті:

| Метод | Що показує |
|---|---|
| `df.head(n)` | Перші n рядків — візуальне розуміння структури |
| `df.info()` | Технічний зріз: типи, non-null значення, пам'ять |
| `df.describe()` | Описова статистика: mean, std, min, quartiles, max |

Також корисні:
- `df.shape` — (рядки, стовпці)
- `df.dtypes` — типи кожної колонки
- `df.isnull().sum()` — кількість пропущених значень
- `df['col'].value_counts()` — частота кожного значення

In [ ]:
# --- Технічний зріз датасету цін на продукти ---
print("=" * 60)
df_food.info()
print("=" * 60)

In [ ]:
# --- Описова статистика ---
print("=== Описова статистика числових колонок ===")
df_food.describe()

In [ ]:
# --- Дослідження пропущених значень ---
print("=== Пропущені значення ===")
missing = df_food.isnull().sum()
missing_pct = (missing / len(df_food) * 100).round(1)
print(pd.DataFrame({'Кількість NaN': missing, '% від загального': missing_pct}))
print(f"\n💡 'admin1' (область) відсутня у {missing_pct['admin1']}% рядків — ці записи відносяться до 'National Average'")

In [ ]:
# --- Унікальні значення категоріальних колонок ---
print("=== Унікальні категорії товарів ===")
print(df_food['category'].value_counts())
print(f"\n=== Топ-10 товарів за кількістю спостережень ===")
print(df_food['commodity'].value_counts().head(20))

### 🎯 Міні-завдання 1

1. Знайдіть кількість унікальних ринків (`market`) у датасеті
2. Визначте, скільки регіонів (`admin1`) охоплює датасет
3. Яка мінімальна та максимальна ціна в датасеті? Це виглядає реалістично?

> **Очікуваний інсайт:** Значна частина записів — це «National Average» (без прив'язки до конкретної локації). Це важливо враховувати при регіональному аналізі.

In [ ]:
# 🎯 Ваш код тут:


---

## 6. Фільтрація даних: loc та query 🎯 <a id='6-filtering'></a>

### Принцип: відсікай зайве

Думати таблицями — це вміти залишати лише те, що потрібно.

### `.loc[]` — фільтрація через булеві маски

```python
# Синтаксис для Series (1 аргумент — рядки):
s.loc[умова_рядків]

# Синтаксис для DataFrame (2 аргументи — рядки + стовпці):
df.loc[умова_рядків, ['col1', 'col2']]  # вибір рядків І стовпців

# ТІЛЬКИ .loc дозволяє модифікувати дані:
df.loc[df['price'] < 10, 'price_flag'] = 'low'
```

**Як працює під капотом:**
При умові `(df['a'] > 100) & (df['b'] < 700)` Pandas **створює окремий тимчасовий булевий масив** для кожної умови, а потім об'єднує їх через `&`. На 1 млн рядків і 3 умовах — це мільйони тимчасових записів у пам'яті.

### `.query()` — SQL-подібний синтаксис через рушій numexpr

```python
df.query('price > 50 and category == "Vegetables"')
```

**Як працює під капотом:**
Метод `.query()` передає текстовий рядок спеціальній бібліотеці **`numexpr`**, яка компілює та виконує всі логічні операції **«на льоту»** без створення проміжних масивів. Це кардинально зменшує споживання RAM при складних умовах на великих даних.

⚠️ На малих даних (<10k рядків) парсинг рядка в `.query()` дає накладні витрати — тому `.loc` може бути швидшим.

### Архітектурне порівняння: коли що обирати?

| Критерій | `.loc[]` | `.query()` |
|---|---|---|
| **Синтаксис** | Булева маска (повторення імені df) | SQL-рядок (лаконічно) |
| **Пам'ять** | ❌ Тимчасові масиви для кожної умови | ✅ `numexpr` — без зайвих масивів |
| **Швидкість** | Краще на малих даних (<10k рядків) | Краще на великих даних |
| **Вибір стовпців** | ✅ `df.loc[умова, ['a','b']]` | ❌ Завжди всі стовпці |
| **Модифікація даних** | ✅ `df.loc[умова, 'col'] = val` | ❌ Тільки читання |

> 💡 **Правило:** `query()` — для читання великих датасетів зі складними умовами. `loc` — для запису або коли потрібно одночасно відібрати конкретні стовпці.

In [ ]:
# --- Фільтрація: тільки ринки в регіонах (без 'National Average') ---
df_regions = df_food.loc[df_food['admin1'].notna()].copy()
print(f"Записів з регіональною прив'язкою: {len(df_regions):,}")
print(f"Унікальних регіонів: {df_regions['admin1'].nunique()}")
print(df_regions['admin1'].value_counts().head(27))

In [ ]:
# --- .loc: вибір рядків І стовпців одночасно ---
# Знайдемо дані по хлібу з 2022 року
mask_bread = df_food['commodity'].str.contains('Bread|Хліб', case=False, na=False)
mask_2022 = df_food['date'].dt.year >= 2022

df_bread_2022 = df_food.loc[mask_bread & mask_2022, ['date', 'admin1', 'commodity', 'price']]
print(f"Записів про хліб з 2022 року: {len(df_bread_2022):,}")
df_bread_2022.head(8)

In [ ]:
# --- .query() — більш читабельний синтаксис ---
# Знайдемо товари категорії 'cereals and tubers' дорожче 50 UAH
df_expensive_cereals = df_food.query(
    "category == 'cereals and tubers' and price > 50 and currency == 'UAH'"
)
print(f"Дорогі зернові (>50 UAH): {len(df_expensive_cereals):,} записів")
print("\nТоп-5 найдорожчих:")
df_expensive_cereals.nlargest(5, 'price')[['date', 'commodity', 'price', 'admin1']]

In [ ]:
# --- Порівняння продуктивності: .loc vs .query ---
# На великих датасетах query() швидший
import time

start = time.perf_counter()
result_loc = df_food.loc[(df_food['price'] > 30) & (df_food['currency'] == 'UAH')]
time_loc = time.perf_counter() - start

start = time.perf_counter()
result_query = df_food.query("price > 30 and currency == 'UAH'")
time_query = time.perf_counter() - start

print(f".loc   : {time_loc*1000:.2f} ms, результатів: {len(result_loc):,}")
print(f".query : {time_query*1000:.2f} ms, результатів: {len(result_query):,}")
print(f"\n✅ Однаковий результат: {len(result_loc) == len(result_query)}")

### 🎯 Міні-завдання 2

1. Відфільтруй записи лише для категорії `'vegetables and fruits'` після 2020 року
2. Знайди записи, де ціна перевищує середню ціну всього датасету
3. Використай `.query()` для фільтрації конкретного товару в конкретному регіоні

> **Очікуваний інсайт:** Після 2022 року ціни на овочі та зернові різко зросли — це відображає реальний вплив повномасштабного вторгнення на продовольчу безпеку.

In [ ]:
# 🎯 Ваш код тут:


---

## 7. Групування та агрегації: Split-Apply-Combine 📊 <a id='7-groupby'></a>

### Парадигма: погляд «згори»

Аналітика часто вимагає погляду на агреговані дані. У `pandas` це реалізовано через парадигму **Split → Apply → Combine**:

```
1. SPLIT    → groupby('category')  → розбити на групи
2. APPLY    → .mean()              → порахувати в кожній групі
3. COMBINE  → підсумкова таблиця   → зшити результати назад
```

### Архітектурна деталь: `groupby()` — ліниве виконання 🦥

Виклик `df.groupby('col')` **не виконує жодних обчислень одразу!**

Він лише створює спеціальний **`DataFrameGroupBy`** об'єкт — словник індексів (покажчиків на рядки, які належать до кожної групи). Реальні обчислення стартують тільки коли ти викликаєш агрегаційну функцію (`.mean()`, `.sum()`). Це суттєво економить пам'ять.

### Чотири шляхи на стадії Apply

| Метод | Що робить | Коли використовувати |
|---|---|---|
| `.agg()` | Зменшує групу до одного числа | Статистика: mean, sum, count, max |
| `.transform()` | Зберігає форму — повертає той самий розмір | Нормалізація в межах групи |
| `.filter()` | Відкидає або залишає цілі групи | Видалення рідкісних категорій |
| `.apply()` | Будь-яка довільна функція | Складна логіка, що не вписується в agg |

### Метод `.agg()` — кілька агрегацій одночасно

```python
df.groupby('category').agg(
    середня_ціна=('price', 'mean'),
    макс_ціна=('price', 'max'),
    унікальних_товарів=('commodity', 'nunique')
)
```

In [ ]:
# --- Базова агрегація: середня ціна за категорією товару ---
df_uah = df_food.query("currency == 'UAH'")  # тільки гривневі ціни

avg_by_category = (
    df_uah
    .groupby('category')['price']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
print("=== Середня ціна за категорією (UAH) ===")
print(avg_by_category)

In [ ]:
# --- Множинна агрегація через .agg() ---
stats_by_category = (
    df_uah
    .groupby('category')
    .agg(
        середня_ціна=('price', 'mean'),
        мін_ціна=('price', 'min'),
        макс_ціна=('price', 'max'),
        кількість_спостережень=('price', 'count'),
        унікальних_товарів=('commodity', 'nunique')
    )
    .round(2)
    .sort_values('середня_ціна', ascending=False)
)
print("=== Статистика за категоріями ===")
stats_by_category

In [ ]:
# --- Аналіз цінових тенденцій по роках ---
# Додаємо колонку 'рік' для групування
df_uah_copy = df_uah.copy()
df_uah_copy['рік'] = df_uah_copy['date'].dt.year

# Середня ціна ХЛІБА по роках
bread_yearly = (
    df_uah_copy
    .query("commodity.str.contains('Bread', case=False)")
    .groupby('рік')['price']
    .agg(
        середня=('mean'),
        медіана=('median')
    )
    .round(2)
)
print("=== Ціна хліба по роках (UAH) ===")
print(bread_yearly)
print(f"\n📈 Зріст ціни хліба з {bread_yearly.index.min()} по {bread_yearly.index.max()}:")
price_start = bread_yearly['середня'].iloc[0]
price_end = bread_yearly['середня'].iloc[-1]
print(f"   {price_start:.2f} → {price_end:.2f} UAH (+{(price_end/price_start - 1)*100:.0f}%)")

In [ ]:
# --- Групування за кількома ключами: рік + категорія ---
# Порівняємо як змінювалися ціни в різних категоріях
df_uah_copy['рік'] = df_uah_copy['date'].dt.year

yearly_category = (
    df_uah_copy
    .groupby(['рік', 'category'])['price']
    .mean()
    .round(2)
    .unstack()  # перетворює категорії у стовпці
)
print("=== Середня ціна за категорією по роках ===")
yearly_category.tail(5)  # останні 5 років

In [ ]:
# --- Демонстрація .transform(): нормалізація всередині групи ---
# transform() повертає Series того ж розміру, що й вхідний DataFrame
# Корисно для: нормалізація, заповнення NaN груповим середнім, Z-score

df_uah_copy2 = df_uah.copy()

# Розраховуємо середню ціну в кожній групі (категорії) та додаємо як нову колонку
df_uah_copy2['середня_по_категорії'] = (
    df_uah_copy2.groupby('category')['price'].transform('mean').round(2)
)
# Відхилення від середнього в категорії
df_uah_copy2['відхилення_%'] = (
    (df_uah_copy2['price'] / df_uah_copy2['середня_по_категорії'] - 1) * 100
).round(1)

print("=== .transform() — ціна відносно середньої по категорії ===")
print("Перші 8 рядків з новими колонками:")
print(df_uah_copy2[['category', 'commodity', 'price', 'середня_по_категорії', 'відхилення_%']].head(8))
print("\n💡 transform() зберігає розмір DataFrame — кожен рядок отримує агрегат своєї групи")

In [ ]:
# --- Демонстрація .filter(): відкидаємо цілі групи ---
# filter() відкидає або залишає цілі групи на основі логічного критерію

# Залишимо лише ті категорії, де кількість спостережень > 1000
df_uah_filtered = (
    df_uah
    .groupby('category')
    .filter(lambda group: len(group) > 1000)
)

print("=== .filter() — лише категорії з >1000 спостережень ===")
print("До filter:")
print(df_uah['category'].value_counts())
print("\nПісля filter (рідкісні категорії видалено):")
print(df_uah_filtered['category'].value_counts())
print(f"\nВидалено рядків: {len(df_uah) - len(df_uah_filtered):,}")

### 🎯 Міні-завдання 3

1. Знайдіть топ-5 найдорожчих товарів (за середньою ціною) в 2023 році
2. Порівняйте середню ціну `Potatoes` до і після 2022 року
3. Скільки унікальних товарів є в кожній категорії?

> **Очікуваний інсайт:** Ціни на товари після 2022 року зросли нерівномірно — деякі категорії (олія, м'ясо) показали більш різкий стрибок, ніж інші.

In [ ]:
# 🎯 Ваш код тут:


---

## 8. Робота з датами (datetime) 📅 <a id='8-datetime'></a>

### Часові ряди — специфічний тип табличних даних

Звичайний текст не дозволяє аналізувати час. Після конвертації у `datetime64` з'являється потужний аксесор **`.dt`**, який дозволяє витягувати будь-яку складову дати:

| Аксесор | Повертає |
|---|---|
| `.dt.year` | Рік |
| `.dt.month` | Місяць (1–12) |
| `.dt.month_name()` | Назва місяця |
| `.dt.dayofweek` | День тижня (0=Пн, 6=Нд) |
| `.dt.quarter` | Квартал (1–4) |
| `.dt.to_period('M')` | Перетворення в Period (рік-місяць) |

### Ресемплінг (resample)

Метод `.resample()` — це `groupby` для часових рядів. Дозволяє агрегувати дані по часових інтервалах (тижень, місяць, рік).

In [ ]:
# --- Перевірка типу дати ---
print(f"Тип колонки 'date': {df_food['date'].dtype}")
print(f"\nПриклад значення: {df_food['date'].iloc[0]}")
print(f"Тип одного значення: {type(df_food['date'].iloc[0])}")

In [ ]:
# --- Аксесор .dt: розбиваємо дату на складові ---
df_time = df_uah.copy()
df_time['рік'] = df_time['date'].dt.year
df_time['місяць'] = df_time['date'].dt.month
df_time['назва_місяця'] = df_time['date'].dt.month_name()
df_time['квартал'] = df_time['date'].dt.quarter

print("=== Результат розбивки дати ===")
df_time[['date', 'рік', 'місяць', 'назва_місяця', 'квартал']].drop_duplicates().head(8)

In [ ]:
# --- Аналіз курсу гривні в часі ---
print("=== Курс UAH/USD по роках ===")
df_rates['рік'] = df_rates['date'].dt.year

yearly_rate = (
    df_rates
    .groupby('рік')['uah_per_usd']
    .mean()
    .round(2)
)
print(yearly_rate.tail(15))

In [ ]:
# --- Ресемплінг: середня ціна хліба по місяцях ---
df_bread = (
    df_uah
    .query("commodity.str.contains('Bread', case=False) and admin1.isna()")
    [['date', 'commodity', 'price']]
    .set_index('date')
    .sort_index()
)

# Ресемплінг по кварталах
bread_quarterly = (
    df_bread
    .resample('QE')  # QE = кінець кварталу
    ['price']
    .mean()
    .round(2)
    .reset_index()
)
bread_quarterly.columns = ['квартал', 'середня_ціна_хліба']
print("=== Середня ціна хліба по кварталах ===")
print(bread_quarterly.tail(12))

In [ ]:
# --- Пошук аномалій у часовому ряді ---
# Знайдемо квартали з найбільшим зростанням ціни
bread_quarterly['зміна_%'] = (
    bread_quarterly['середня_ціна_хліба'].pct_change() * 100
).round(1)

print("=== Квартали з найбільшим зростанням цін на хліб ===")
print(bread_quarterly.nlargest(5, 'зміна_%')[['квартал', 'середня_ціна_хліба', 'зміна_%']])
print("\n💡 Знайдіть квартали, що збігаються з ключовими подіями в Україні")

### 🎯 Міні-завдання 4

1. Визначте, в якому місяці (1–12) ціни на овочі найвищі впродовж року
2. Порівняйте середній курс UAH/USD до і після 2022 року
3. Знайдіть момент (місяць/рік), коли відбулося найбільше стрибкоподібне зростання курсу

> **Очікуваний інсайт:** Курс гривні корелює з цінами на імпортовані товари — девальвація безпосередньо відображається у споживчих цінах.

In [ ]:
# 🎯 Ваш код тут:


---

## 9. Об'єднання датасетів: merge та concat 🔗 <a id='9-merge'></a>

### Це найважливіша навичка! 🔥

У реальному світі дані розкидані по різних таблицях. Вміння їх зшивати — ключова аналітична навичка.

### Горизонтальне об'єднання: `pd.merge()` → аналог SQL JOIN

| `how=` | Аналог SQL | Поведінка |
|---|---|---|
| `'inner'` | INNER JOIN | Тільки рядки, що є в обох таблицях |
| `'left'` | LEFT JOIN | Всі рядки з лівої + збіги з правої |
| `'right'` | RIGHT JOIN | Всі рядки з правої + збіги з лівої |
| `'outer'` | FULL OUTER JOIN | Всі рядки з обох, NaN де немає пари |

### Вертикальне об'єднання: `pd.concat()` → «поставити таблиці одна під одною»

```python
pd.concat([df_2022, df_2023], ignore_index=True)  # склеїти по рядках
```

In [ ]:
# --- Підготовка: середня ціна хліба по місяцях (National Average) ---
bread_monthly = (
    df_food
    .query(
        "commodity.str.contains('Bread', case=False) "
        "and admin1.isna() "
        "and currency == 'UAH'"
    )
    .assign(
        рік=lambda x: x['date'].dt.year,
        місяць=lambda x: x['date'].dt.month
    )
    .groupby(['рік', 'місяць'])['price']
    .mean()
    .round(2)
    .reset_index()
)
bread_monthly.columns = ['рік', 'місяць', 'ціна_хліба_uah']
print(f"Місячних записів про ціну хліба: {len(bread_monthly)}")
bread_monthly.tail()

In [ ]:
# --- Підготовка: курс гривні по місяцях ---
rates_monthly = df_rates[['date', 'year', 'uah_per_usd']].copy()
rates_monthly.columns = ['date', 'рік', 'курс_uah_usd']
rates_monthly['місяць'] = rates_monthly['date'].dt.month
rates_monthly = rates_monthly[['рік', 'місяць', 'курс_uah_usd']]
print(f"Місячних записів курсу: {len(rates_monthly)}")
rates_monthly.tail()

In [ ]:
# --- MERGE: об'єднуємо ціни хліба та курс UAH/USD ---
# Ключ злиття: рік + місяць
df_bread_rates = pd.merge(
    bread_monthly,        # ліва таблиця
    rates_monthly,        # права таблиця
    on=['рік', 'місяць'], # спільні ключі
    how='inner'           # тільки записи, що є в обох
)

print(f"=== Результат злиття ===")
print(f"Рядків: {len(df_bread_rates)} (перетин дат)")
df_bread_rates.tail(8)

In [ ]:
# --- Аналіз: ціна хліба в доларах через злиття ---
df_bread_rates['ціна_хліба_usd'] = (
    df_bread_rates['ціна_хліба_uah'] / df_bread_rates['курс_uah_usd']
).round(3)

print("=== Ціна хліба в UAH та USD по роках ===")
yearly_comparison = (
    df_bread_rates
    .groupby('рік')[['ціна_хліба_uah', 'курс_uah_usd', 'ціна_хліба_usd']]
    .mean()
    .round(3)
)
print(yearly_comparison)
print("""
💡 Інсайт: навіть коли ціна в гривні зростає, ціна в доларах може
   залишатися стабільною або навіть падати — через девальвацію.
   Це показує реальну купівельну спроможність населення.""")

In [ ]:
# --- CONCAT: вертикальне об'єднання двох датасетів ---
# Розіб'ємо дані на «до 2022» та «після 2022» і знову склеїмо
df_before_2022 = df_bread_rates.query("рік < 2022").copy()
df_after_2022 = df_bread_rates.query("рік >= 2022").copy()

df_before_2022['епоха'] = 'до 2022'
df_after_2022['епоха'] = 'після 2022'

df_combined = pd.concat([df_before_2022, df_after_2022], ignore_index=True)

print(f"до 2022: {len(df_before_2022)} рядків")
print(f"після 2022: {len(df_after_2022)} рядків")
print(f"після concat: {len(df_combined)} рядків")

# Порівняння середніх по епохах
print("\n=== Порівняння між епохами ===")
df_combined.groupby('епоха')[['ціна_хліба_uah', 'курс_uah_usd', 'ціна_хліба_usd']].mean().round(3)

In [ ]:
# --- Демонстрація різних типів JOIN ---
# Підготуємо маленькі датасети для наочності
df_prices_small = pd.DataFrame({
    'рік': [2019, 2020, 2021, 2022],
    'ціна_хліба': [12.5, 15.0, 18.3, 26.7]
})
df_rates_small = pd.DataFrame({
    'рік': [2020, 2021, 2022, 2023],  # 2019 відсутній, 2023 — зайвий
    'курс': [26.9, 27.3, 32.3, 36.8]
})

print("INNER JOIN (перетин — тільки 2020-2022):")
print(pd.merge(df_prices_small, df_rates_small, on='рік', how='inner'))

print("\nLEFT JOIN (всі роки лівої таблиці, 2019 без курсу):")
print(pd.merge(df_prices_small, df_rates_small, on='рік', how='left'))

print("\nOUTER JOIN (всі роки, NaN де немає пари):")
print(pd.merge(df_prices_small, df_rates_small, on='рік', how='outer'))

### 🎯 Фінальне завдання

Проведіть комплексний аналіз:

1. Завантажте `global-market-monitor.csv` та відфільтруйте дані тільки по Україні
2. Знайдіть, які продукти показали найбільший річний приріст ціни (`YoYChangeMonth`)
3. Об'єднайте ці дані з курсом UAH/USD через `pd.merge()` по даті/року
4. Дайте відповідь: **чи корелює девальвація гривні зі зростанням цін на продукти?**

> **Очікуваний інсайт:** Аналіз злитих датасетів дозволяє побачити макроекономічні залежності, які не видно з жодної таблиці окремо — це і є справжня сила pandas як аналітичного інструменту.

In [ ]:
# --- Крок 1: Завантажуємо глобальний моніторинг ринку ---
df_global = pd.read_csv(
    os.path.join(DATA_DIR, 'global-market-monitor.csv'),
    usecols=['Date', 'CountryName', 'MainStapleFood', 'YoYChangeMonth', 'PriceTrendMonth'],
    parse_dates=['Date']
)

# Відфільтровуємо Україну
df_ukraine_global = (
    df_global
    .query("CountryName.str.strip() == 'Ukraine'")
    .copy()
)
df_ukraine_global.columns = ['date', 'країна', 'продукт', 'зміна_рік_%', 'тренд']
df_ukraine_global['продукт'] = df_ukraine_global['продукт'].str.strip()

print(f"Записів по Україні: {len(df_ukraine_global)}")
print(f"Продукти: {df_ukraine_global['продукт'].unique()}")
df_ukraine_global.head()

In [ ]:
# --- Крок 2: Найбільший річний приріст цін ---
worst_increases = (
    df_ukraine_global
    .dropna(subset=['зміна_рік_%'])
    .nlargest(10, 'зміна_рік_%')
    [['date', 'продукт', 'зміна_рік_%', 'тренд']]
)
print("=== Топ-10 найбільших річних стрибків цін ===")
worst_increases

In [ ]:
# --- Крок 3: Об'єднання з курсом валют ---
df_ukraine_global['рік'] = df_ukraine_global['date'].dt.year
df_ukraine_global['місяць'] = df_ukraine_global['date'].dt.month

df_merged_full = pd.merge(
    df_ukraine_global,
    rates_monthly,
    on=['рік', 'місяць'],
    how='left'
)

print(f"Після злиття: {len(df_merged_full)} рядків")
print(f"Пропущених курсів: {df_merged_full['курс_uah_usd'].isna().sum()}")
df_merged_full.dropna(subset=['зміна_рік_%', 'курс_uah_usd'], inplace=True)
df_merged_full.tail(6)

In [ ]:
# --- Крок 4: Кореляція курсу та цін ---
correlation = df_merged_full[['зміна_рік_%', 'курс_uah_usd']].corr()
print("=== Кореляція: зміна ціни (%) ~ курс UAH/USD ===")
print(correlation)

corr_value = correlation.loc['зміна_рік_%', 'курс_uah_usd']

print(f"""
📊 Кореляція = {corr_value:.3f}

Інтерпретація:
  • > 0.7 → сильна пряма кореляція (девальвація → дорожче)
  • 0.3–0.7 → помірна кореляція
  • < 0.3 → слабка кореляція

💡 Середня ціна за курсовими діапазонами:
""")

# Групуємо по курсовим діапазонам
bins = [0, 20, 27, 32, 40, 100]
labels = ['<20', '20-27', '27-32', '32-40', '>40']
df_merged_full['курс_діапазон'] = pd.cut(df_merged_full['курс_uah_usd'], bins=bins, labels=labels)
print(df_merged_full.groupby('курс_діапазон')['зміна_рік_%'].mean().round(1))

---

## 🏁 Підсумок уроку

### Що ми навчились думати:

```
ДАНІ  →  РОЗУМІННЯ  →  ТРАНСФОРМАЦІЯ  →  ІНСАЙТ
 CSV      .info()        groupby()      кореляція
         describe()       merge()       тренди
```

### Ключові концепції:

| Концепція | Суть |
|---|---|
| **DataFrame** | Модель реального світу: рядки = спостереження, стовпці = атрибути |
| **Series** | 1D масив з індексом — базовий блок DataFrame |
| **Data Alignment** | Операції зіставляють дані за мітками, а не позицією |
| **Category dtype** | Економія до 90% пам'яті для повторюваних текстів |
| **loc vs query** | `loc` — булева маска; `query` — SQL-подібний рядок |
| **groupby** | Split → Apply → Combine |
| **merge vs concat** | Горизонтальне (JOIN) vs вертикальне (склеювання) |
| **.dt accessor** | Витягування складових дати з datetime64 |

### Наступний крок

У **Уроці 52 (частина 2)** ми візуалізуємо ці дані через **Matplotlib та Plotly**

---
*📚 Урок 52 — Модуль 3: Data Analysis & Visualization*